## The NOAA Coastal Ocean Reanalysis (CORA) dataset outputs are referenced to mean sea level. This notebook allows users to download CORA data from the NOAA Open Data Dissemination (NODD), save the data as a .csv file and then use the file(s) with NOAA’s Tidal Analysis Datum Calculator (TADC) python script to convert data to other tidal datums. If you already have .csv files available, you can skip to the last couple of cell blocks of the script to the part where the TADC script is used with your files.
***
## **The TADC script (SDC.py) and it's associated config.cfg file are provided in the TADC_Files folder. A sample file is provided in the Sample_Files folder.** 

### First, import any necessary python modules. If a module is not found, you will need to pip (or conda) install the module.

## To access data on the NODD with this notebook, you must be sure you are using a version of zarr < 3.0.0. If your current installation of zarr is version 3, please pip install zarr==2.18.7.

In [ ]:
import requests
import shutil
import os
import subprocess
import pandas as pd
import glob
from natsort import natsorted
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import dask
import intake
import intake_xarray
import s3fs
import holoviews as hv
import geoviews as gv
import hvplot.xarray
import holoviews.operation.datashader as dshade
from bokeh.models import DatetimeTickFormatter, HoverTool
import cmocean
from bokeh.resources import INLINE
import bokeh.io
from bokeh import *
bokeh.io.output_notebook(INLINE)
hv.extension('bokeh')

# ***<span style='color:Blue'> If you are using files of previously acquired data and are not acquiring new CORA data, skip down to the last 2 cell blocks of code where you can use your data with the Tidal Analysis Datum Calculator script. If you are acquiring new CORA data, continue through the following cell blocks of code. </span>***

### Access the data on the NODD and initialize the available CORA datasets.

### *This accesses a .yml file located on the NODD that shows which CORA output files are available to import.*

In [ ]:
# @title This accesses a .yml file located on the NODD that shows which CORA output files are available to import.
catalog = intake.open_catalog("s3://noaa-nos-cora-pds/CORA_V1.1_intake.yml",storage_options={'anon':True})
list(catalog)

#### ***CORA-V1.1-fort.63:*** Hourly water levels-- ***'zeta'*** is the water elevation variable referenced to mean sea level
#### ***CORA-V1.1-swan_DIR.63:*** Hourly mean wave direction-- ***'swan_DIR'*** is the mean wave direction variable
#### ***CORA-V1.1-swan_TPS.63:*** Hourly peak wave periods-- ***'swan_TPS'*** is the peak wave period variable
#### ***CORA-V1.1-swan_HS.63:*** Hourly significant wave heights-- ***'swan_HS'*** is the significant wave height variable
#### ***CORA-V1.1-Grid:*** Hourly water levels interpolated from model nodes to 500-meter resolution coastal grid-- ***'zeta'*** is the water elevation variable <br>
>#### All datasets denoted as **'-timeseries'** are optimized for pulling long time series (greater than a few days)
>#### For up to a few days of data, use the regular dataset (not labeled **'-timeseries'** in the catalog description)

### Using the .to_dask() command with the water level dataset located in the catalog will create an xarray dataset. There is data at 1,813,443 model nodes spanning 385,704 hours (44 years, 1979-2022). The 'zeta' hourly water level variable is given in dimensions of time and node.

In [ ]:
ds = catalog["CORA-V1.1-fort.63-timeseries"].to_dask()
ds

### *Which points would you like to query?* 

### Create a pandas dataframe with locations where you would like to pull CORA data. You could bring in your own file of locations using pd.read_csv('yourfile.csv'). For this example, we are pulling in coordinates for NOAA NWLON stations using the CO-OPS API.

In [ ]:
units= 'metric'

station_type = 'waterlevels'

server = 'https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations/' # accessing NOAA CO-OPS api

myurl = (server + '.json?type='+station_type+'&units='+units)

urlResponse = requests.get(myurl)
content=urlResponse.json()

stations = content['stations']
stations_df = pd.DataFrame(stations)

stations_df = stations_df[['id','lat','lng']]
station_id = ['8665530'] # Using NOAA NWLON stations

stations_df = stations_df[stations_df['id'].isin(station_id)]

stations_df

### A function is created to find the nearest node to a given set of coordinates.

In [ ]:
# find the indices of the points in (x,y) closest to the points in (xi,yi)
def nearxy(x,y,xi,yi):
    ind = np.ones(len(xi),dtype=int)
    for i in range(len(xi)):
        dist = np.sqrt((x-xi[i])**2+(y-yi[i])**2)
        ind[i] = dist.argmin()
    return ind

### Run the function in a loop to create a list of node indices where you want data.

In [ ]:
x = ds['x'].values
y = ds['y'].values

nodeidx = []
for i in range(0,len(stations_df)):
    ind = nearxy(x,y,[stations_df.lng.iloc[i]],[stations_df.lat.iloc[i]])
    nodeidx.append(ind)

nodeidx = [item for sublist in nodeidx for item in sublist]

### Extract data for a specified time range, create dataframes for saving the data, then create plots using hvplot. Here, the hourly water levels are accessed for the specified node indices and time range. Using .compute() calls the dataset into memory, so this can take awhile depending on the length of data and number of nodes you are requesting.

In [ ]:
%%time
start="1983-01-01 00:00:00"
end="2001-12-31 23:00:00"

zeta_tslice = ds['zeta'].sel(time=slice(start, end),node=nodeidx).compute()

### Plot your time series data.

In [ ]:
import panel as pn

tickfmt = DatetimeTickFormatter(years="%m/%d/%Y", months="%m/%d/%Y", days = '%m/%d/%Y', hourmin = '%H:%M')
tooltips = [
    ("time", "@time{%F %T}"),
    ("water level", "@zeta"),
]
hover = HoverTool(tooltips=tooltips,formatters={
        '@time': 'datetime'})

plots = []
for i in range(0,len(stations_df)):

    plot = zeta_tslice[:,i].hvplot(x='time', grid=True, xformatter=tickfmt, tools=[hover])

    plots.append(plot)

pn.Column(*plots)

### Save the CORA data at each location as a .csv file. To save each file, the data that was acquired at each location is stored in a pandas dataframe and then written to a .csv file using pd.to_csv().

In [ ]:

dt_range = pd.date_range(start, end, freq='h',inclusive='both')
zeta_df = pd.DataFrame({'time': dt_range})

filepath = 'WorkingDirectory\\Sample_Files\\' # Sample_Files is provided in the Github repository

for i in range(0,len(stations_df)):
    zeta_df['zeta'] = zeta_tslice[:,i]
    zeta_df = zeta_df.round(3) # round the wl digits to 3 for use with the TAD calculator
    zeta_df['time'] = pd.to_datetime(zeta_df['time'])
    zeta_df['time'] = zeta_df['time'].dt.strftime('%m/%d/%Y %H:%M') # change the datetime format for use with the TAD calculator

    wse_filename = str(stations_df.id.iloc[i]) + '_CORA' + '.csv' # create filename with station id and time range
    zeta_df[['time',"zeta"]].to_csv(filepath+wse_filename,index=False) # write the files to .csv

# ***<span style='color:Blue'> Start here if you are using files of previously acquired data. Below, point to your file directory with the filepath variable. </span>***

### Loop through your CORA .csv files and run the data through the TADC using subprocess with the SDC.py file in the TADC_Files folder. The config.cfg file is altered on each loop and used by SDC.py. Two output files will be written out for each file (the tidal datums .out file and a Highs/Lows .csv file).

In [ ]:
# Specify the path to the folder containing the CSV files and use glob to iterate through the files
configpath = 'WorkingDirectory\\TADC_Files\\' # TADC_Files is provided in the Github repository
filepath = 'WorkingDirectory\\Sample_Files\\' # Sample_Files is provided in the Github repository
csv_path = filepath + '*.csv'

for fname in natsorted(glob.glob(csv_path)): # natsorted makes sure of the natural sorting of files
    
    file_name = os.path.split(fname)[-1]  # Extract the filename from the path
    station = int(file_name[0:7])
    with open(configpath + 'config.cfg', "r") as file:
        lines = file.readlines()
    
        lines[22] = 'fname = ' + filepath + file_name + '\n'
        lines[47] = 'subordinate_lon = ' + str(station) + '\n'
        lines[51] = 'subordinate_lat = ' + str(station) + '\n'
    
    with open(configpath + 'config.cfg', "w") as file:
        file.writelines(lines)

    with open(configpath + 'SDC.py', "r") as file:
        SDClines = file.readlines()

        SDClines[158] = 'CONFIG_FILE = ' + 'r"' + configpath + 'config.cfg"' + ' \n'

    with open(configpath + 'SDC.py', "w") as file:
        file.writelines(SDClines)
        
    proc = subprocess.run(['python', configpath + 'SDC.py'], capture_output=True, text=True)


### This last part parses the tidal datums elevation table from the .out file and writes it to a new .csv file.

In [ ]:
# datastart should match the start date of your CORA water level data file; If using multiple locations, all need to start on same date for this code to work properly
start="1983-01-01 00:00:00" # all files that you are looping through should start at the same time
datastart = "Data Start:  " + start
outputpath = filepath + 'Outputs\\'
folder_path = outputpath + '*.out'

fulldata = []

for fname in natsorted(glob.glob(folder_path)): # natsorted makes sure of the natural sorting of files

    file_name = os.path.split(fname)[-1]  # Extract the filename from the path
    out_path = outputpath + file_name

    with open(out_path, "r") as file:

        for line_number, line in enumerate(file, start=1):
            if datastart in line:
                fulldata.append(int(file_name[0:7])) 
                break  # Stop after finding the first match
    
    with open(out_path, "r") as file:
        
        lines = len(file.readlines())
        
    with open(out_path, "r") as file:
        
        for line_number, line in enumerate(file, start=1):
            
            if "HWL" in line:                   
                    
                colspecs = [(0,5),(7,15)]
                df_fwf = pd.read_fwf(out_path, colspecs=colspecs,skiprows=line_number - 1, skipfooter=lines - line_number - 12, names=['Datums','meters'])
                                
                grid_fname = outputpath + 'Datums_' + file_name + '.csv'
                df_fwf.to_csv(grid_fname,index=False) # write the files to .csv

                break  